In [1]:
from pyspark.sql import functions as F

# ============================================================
# NB_Gold_Dim_Date
# ============================================================

TARGET_TABLE = "dim_date"

START_DATE = "2020-01-01"
END_DATE = "2035-12-31"

df_date = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit(START_DATE)),
                F.to_date(F.lit(END_DATE)),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("date")
    )
)

df_gold = (
    df_date
    .withColumn(
        "date_key",
        F.date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn(
        "is_weekend",
        F.dayofweek("date").isin([1, 7])
    )
    .withColumn(
        "year_month",
        F.date_format("date", "yyyy-MM")
    )
    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "month_name",
        "week_of_year",
        "day_of_month",
        "day_of_week",
        "day_name",
        "is_weekend",
        "year_month"
    )
)

df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(TARGET_TABLE)

print(f"Gold table created: {TARGET_TABLE}")
print(f"Records: {df_gold.count()}")

df_gold.show(10, truncate=False)

StatementMeta(, 3ab9a179-db6b-46ed-898f-e8a0fd3062cc, 3, Finished, Available, Finished, False)

Gold table created: dim_date
Records: 5844
+--------+----------+----+-------+-----+----------+------------+------------+-----------+---------+----------+----------+
|date_key|date      |year|quarter|month|month_name|week_of_year|day_of_month|day_of_week|day_name |is_weekend|year_month|
+--------+----------+----+-------+-----+----------+------------+------------+-----------+---------+----------+----------+
|20200101|2020-01-01|2020|1      |1    |January   |1           |1           |4          |Wednesday|false     |2020-01   |
|20200102|2020-01-02|2020|1      |1    |January   |1           |2           |5          |Thursday |false     |2020-01   |
|20200103|2020-01-03|2020|1      |1    |January   |1           |3           |6          |Friday   |false     |2020-01   |
|20200104|2020-01-04|2020|1      |1    |January   |1           |4           |7          |Saturday |true      |2020-01   |
|20200105|2020-01-05|2020|1      |1    |January   |1           |5           |1          |Sunday   |true